<a href="https://colab.research.google.com/github/kuds/courtside-dynamics/blob/main/notebooks/humanoid_tennis_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Humanoid Tennis Automatic Curriculum

Train the constrained Unitree G1 tasks as a transfer curriculum: Stage 0 learns a physical racket intercept, Stage 1 warm-starts that policy for a controlled return, and Stage 2 warm-starts it again for randomized returns.

A stage advances only when one independently selected best checkpoint passes the canonical held-out success, per-side, safety, and predecessor-retention gates. The checkpoint's policy parameters and matching observation-normalization statistics initialize the next stage. A fresh optimizer, reward-normalization state, schedule, buffers, callbacks, and timestep counter start at every boundary because the objective changed.

Episode length, mean reward, rally count, and valid returns are reported. They can be enabled as additional gates below, but are not trusted by default: a stalled or reward-hacking policy can make those numbers look good without completing the tennis task.

This remains an experimental curriculum baseline. It does not demonstrate two free-standing humanoids learning full tennis, and Stages 3–5 do not yet have training recipes.

## 1. Install

Install the training and notebook extras. During pull-request review, replace BRANCH-NAME with the branch under test.

In [ ]:
!pip install -q "courtside-dynamics[train,notebooks] @ git+https://github.com/kuds/courtside-dynamics"
# To test an unmerged branch instead:
# !pip install -q "courtside-dynamics[train,notebooks] @ git+https://github.com/kuds/courtside-dynamics@BRANCH-NAME"


## 2. Configure Colab rendering

Run this before importing the environments so MuJoCo selects EGL correctly. In Colab, choose a GPU runtime first. The helper is a no-op outside Colab.

In [ ]:
from courtside_dynamics.colab_setup import setup_colab

setup_colab()


## 3. Configure the curriculum

The default run trains Stages 0, 1, and 2 in order with PPO. AUTO_ADVANCE=False trains only the first listed stage. QUICK_TEST validates plumbing but cannot produce promotion evidence, so it is deliberately incompatible with automatic advancement.

The canonical gate already requires 80% current-stage success, 80% on each mirrored side, 75% retention on every earlier stage and side, 100 unique held-out initial states, and zero unsafe episodes. Optional extra criteria are all disabled by default. For example, setting min_mean_reward adds that requirement; it never replaces canonical task success.

In [ ]:
STAGES = (0, 1, 2)
STAGE_RECIPES = {
    0: "HumanoidTennisStage0Intercept",
    1: "HumanoidTennisStage1AnchoredReturn",
    2: "HumanoidTennisStage2RandomizedReturn",
}

AUTO_ADVANCE = True
ALGO = None  # None preserves the recipe default (PPO)
USE_DRIVE = True
QUICK_TEST = False
SEED = 0
TOTAL_TIMESTEPS = None  # None preserves each stage recipe's budget
N_ENVS = None  # None preserves each recipe's validated n_envs=1
EARLY_STOP_PATIENCE = 20
MODEL_KWARGS = {}
RUN_ORACLE_PREFLIGHT = True
START_TENSORBOARD = False
REPLAY_EACH_STAGE = True
DISCONNECT_WHEN_DONE = True

# These diagnostics become extra AND-gates only when set to a number.
# Longer episodes are not inherently better: successful stages terminate early.
EXTRA_STAGE_CRITERIA = {
    0: {
        "min_mean_reward": None,
        "min_mean_episode_steps": None,
        "max_mean_episode_steps": None,
        "min_mean_valid_returns": None,
        "min_mean_rally_count": None,
    },
    1: {
        "min_mean_reward": None,
        "min_mean_episode_steps": None,
        "max_mean_episode_steps": None,
        "min_mean_valid_returns": None,
        "min_mean_rally_count": None,
    },
    2: {
        "min_mean_reward": None,
        "min_mean_episode_steps": None,
        "max_mean_episode_steps": None,
        "min_mean_valid_returns": None,
        "min_mean_rally_count": None,
    },
}

if not STAGES or STAGES[0] != 0 or STAGES != tuple(range(STAGES[-1] + 1)):
    raise ValueError("STAGES must be a contiguous curriculum beginning at Stage 0.")
if any(stage not in STAGE_RECIPES for stage in STAGES):
    raise ValueError("This notebook supports only implemented Stages 0, 1, and 2.")
if set(EXTRA_STAGE_CRITERIA) != set(STAGES):
    raise ValueError("EXTRA_STAGE_CRITERIA must contain exactly the selected stages.")
if QUICK_TEST and AUTO_ADVANCE:
    raise ValueError("QUICK_TEST is not promotion evidence; set AUTO_ADVANCE=False.")


## 4. Mount Drive and create an immutable curriculum root

Every stage receives a separate child run directory. The root manifest is rewritten after each completed stage, so a promotion stop or runtime interruption leaves an inspectable lineage.

In [ ]:
from pathlib import Path

from courtside_dynamics.notebook_utils import mount_drive, resolve_run_dir
from courtside_dynamics.recipes import RECIPES

if USE_DRIVE:
    mount_drive()

first_recipe = RECIPES[STAGE_RECIPES[STAGES[0]]]
RESOLVED_ALGO = (ALGO or first_recipe.default_algo).upper()
if AUTO_ADVANCE and RESOLVED_ALGO != "PPO":
    raise ValueError("Automatic warm-start advancement currently supports PPO only.")

CURRICULUM_ROOT = Path(
    resolve_run_dir(
        "HumanoidTennisCurriculum",
        RESOLVED_ALGO,
        use_drive=USE_DRIVE,
    )
)
STAGES_TO_RUN = STAGES if AUTO_ADVANCE else STAGES[:1]
print("Curriculum root:", CURRICULUM_ROOT)
print("Stages scheduled:", STAGES_TO_RUN)


## 5. Define one-shot training, evaluation, and audit helpers

The canonical suite is evaluated exactly once for each trained checkpoint. Stage 1 and Stage 2 re-evaluate their current checkpoint on all predecessors; summaries from older policies are never reused as retention evidence.

In [ ]:
import json
from statistics import fmean

import matplotlib.pyplot as plt
from IPython.display import display

from courtside_dynamics.notebook_utils import (
    check_run_artifacts,
    display_video,
    plot_eval_info,
    plot_learning_curve,
    plot_training_health,
    print_stage_summary,
    record_best_model_video,
)
from courtside_dynamics.recipes import build_train_config, make_env_fn
from courtside_dynamics.scripted_policies import (
    run_humanoid_tennis_stage0_oracle,
    run_humanoid_tennis_stage1_oracle,
)
from courtside_dynamics.training import WarmStartConfig, train
from courtside_dynamics.training.algos import resolve_algo
from courtside_dynamics.training.artifacts import (
    RUN_LAYOUT,
    artifact_path,
    locate_artifact,
)
from courtside_dynamics.training.tennis_curriculum import (
    assess_curriculum_promotion,
    evaluate_curriculum_stage,
)


def write_json(path, payload):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2) + "\n")
    temporary.replace(path)


def make_stage_config(stage, stage_dir, warm_start):
    recipe_name = STAGE_RECIPES[stage]
    recipe = RECIPES[recipe_name]
    recipe_n_envs = recipe.extra_cfg.get("n_envs")
    n_envs = N_ENVS if N_ENVS is not None else recipe_n_envs
    if n_envs is None:
        n_envs = 1
    cfg = build_train_config(
        recipe_name,
        algo=RESOLVED_ALGO,
        log_dir=str(stage_dir),
        total_timesteps=TOTAL_TIMESTEPS,
        quick_test=QUICK_TEST,
        seed=SEED + stage * 10_000,
        n_envs=n_envs,
        early_stop_patience=EARLY_STOP_PATIENCE,
        model_kwargs=MODEL_KWARGS,
        warm_start=warm_start,
    )
    if cfg.eval_freq > cfg.total_timesteps:
        raise ValueError("Each stage budget must include at least one evaluation.")

    probe_env = cfg.env_fn()
    try:
        observation, reset_info = probe_env.reset(seed=SEED)
        assert reset_info["curriculum_stage"] == stage
        print(
            f"Stage {stage}: {recipe.description}\n"
            f"  action={probe_env.action_space.shape}, observation={observation.shape}, "
            f"active={reset_info['active_action_count']}/58, "
            f"episode_len={probe_env.episode_len}"
        )
    finally:
        probe_env.close()
    return cfg


def run_oracle_preflight(stage, cfg):
    if not RUN_ORACLE_PREFLIGHT:
        print("Oracle preflight skipped by configuration.")
        return
    if stage == 2:
        print("Stage 2 has randomized feeds and no robust scripted oracle.")
        return
    runner = (
        run_humanoid_tennis_stage0_oracle
        if stage == 0
        else run_humanoid_tennis_stage1_oracle
    )
    oracle_env = cfg.env_fn()
    try:
        results = [
            runner(oracle_env, serving_side=side, seed=SEED)
            for side in ("a", "b")
        ]
    finally:
        oracle_env.close()
    for result in results:
        print(
            f"oracle side={result.serving_side.label}, "
            f"success={result.stage_success}, steps={result.steps}, "
            f"return={result.total_reward:.3f}"
        )
    if not all(result.stage_success for result in results):
        raise RuntimeError("Physical oracle preflight failed; training aborted.")


def render_and_audit_stage(cfg, stage_dir):
    print_stage_summary(stage_dir)
    summary_lines = (stage_dir / "stage_summary.txt").read_text().splitlines()
    status_line = next(
        (line for line in summary_lines if line.startswith("Status:")),
        "",
    )
    if status_line.split(":", 1)[-1].strip() == "interrupted":
        raise RuntimeError(
            "Interrupted training was saved for recovery but cannot auto-promote."
        )
    figures = [
        plot_learning_curve(
            stage_dir,
            save_path=artifact_path(stage_dir, "learning_curve"),
        ),
        # plot_eval_info returns one figure per themed report page.
        *(
            plot_eval_info(
                stage_dir,
                save_path=artifact_path(stage_dir, "eval_headline"),
            )
            or []
        ),
        plot_training_health(
            stage_dir,
            save_path=artifact_path(stage_dir, "training_health_plot"),
        ),
    ]
    for figure in figures:
        if figure is not None:
            plt.close(figure)
    if REPLAY_EACH_STAGE:
        video_path = record_best_model_video(
            stage_dir,
            cfg.env_fn,
            algo=cfg.algo,
            video_length=cfg.video_length,
        )
        display(display_video(video_path))
    else:
        print("Best-model replay skipped by configuration.")

    missing = check_run_artifacts(stage_dir)
    if REPLAY_EACH_STAGE:
        blocking_missing = missing
    else:
        blocking_missing = [
            item for item in missing if item != RUN_LAYOUT["best_model_video"]
        ]
    if blocking_missing:
        raise FileNotFoundError(f"Stage artifacts are incomplete: {blocking_missing}")


def held_out_metrics(summary):
    episodes = summary.episodes
    per_side_success = {}
    for side in ("a", "b"):
        side_episodes = [
            episode
            for episode in episodes
            if episode.condition.serving_side.label == side
        ]
        per_side_success[side] = fmean(
            float(episode.stage_success) for episode in side_episodes
        )
    return {
        "success_rate": summary.success_rate,
        "success_rate_ci95": list(summary.success_rate_ci95),
        "per_side_success_rate": per_side_success,
        "mean_reward": fmean(episode.total_reward for episode in episodes),
        "mean_episode_steps": fmean(episode.steps for episode in episodes),
        "mean_valid_returns": fmean(
            episode.valid_return_count for episode in episodes
        ),
        "mean_rally_count": fmean(episode.rally_count for episode in episodes),
        "valid_return_rate": summary.valid_return_rate,
    }


def extra_criteria_failures(stage, metrics):
    criteria = EXTRA_STAGE_CRITERIA[stage]
    failures = []
    minimums = {
        "min_mean_reward": "mean_reward",
        "min_mean_episode_steps": "mean_episode_steps",
        "min_mean_valid_returns": "mean_valid_returns",
        "min_mean_rally_count": "mean_rally_count",
    }
    for criterion_name, metric_name in minimums.items():
        threshold = criteria[criterion_name]
        if threshold is not None and metrics[metric_name] < threshold:
            failures.append(
                f"{metric_name}={metrics[metric_name]:.3f} is below {threshold:.3f}"
            )
    maximum = criteria["max_mean_episode_steps"]
    if maximum is not None and metrics["mean_episode_steps"] > maximum:
        failures.append(
            f"mean_episode_steps={metrics['mean_episode_steps']:.3f} "
            f"exceeds {maximum:.3f}"
        )
    return failures


def evaluate_for_promotion(stage, cfg, stage_dir):
    located_model = locate_artifact(stage_dir, "best_model")
    located_normalizer = locate_artifact(stage_dir, "best_vec_normalize")
    if located_model is None or located_normalizer is None:
        raise FileNotFoundError(
            "Promotion requires co-produced best_model.zip and "
            "best_vec_normalize.pkl."
        )
    best_model_path = Path(located_model)
    best_normalizer_path = Path(located_normalizer)

    best_policy = resolve_algo(cfg.algo).load(str(best_model_path), device="cpu")
    policy_id = f"{CURRICULUM_ROOT.name}:stage-{stage}:best_model"
    normalization_id = f"{CURRICULUM_ROOT.name}:stage-{stage}:best_vec_normalize"
    summaries = {}
    for eval_stage in range(stage + 1):
        summary = evaluate_curriculum_stage(
            best_policy,
            make_env_fn(STAGE_RECIPES[eval_stage]),
            policy_id=policy_id,
            policy_artifact_path=best_model_path,
            normalization_id=normalization_id,
            normalization_artifact_path=best_normalizer_path,
        )
        summaries[eval_stage] = summary
        write_json(stage_dir / f"held_out_stage_{eval_stage}.json", summary.to_dict())
        low, high = summary.success_rate_ci95
        print(
            f"held-out stage={eval_stage} success={summary.success_rate:.1%} "
            f"CI95=[{low:.1%}, {high:.1%}] "
            f"unique_states={summary.unique_initial_state_count}"
        )

    report = assess_curriculum_promotion(
        summaries[stage],
        prior_stages=tuple(summaries[index] for index in range(stage)),
    )
    write_json(stage_dir / "promotion_report.json", report.to_dict())
    metrics = held_out_metrics(summaries[stage])
    extra_failures = extra_criteria_failures(stage, metrics)
    automatic_eligible = report.eligible_for_manual_promotion and not extra_failures
    decision = {
        "automatic_eligible": automatic_eligible,
        "canonical_gate_passed": report.eligible_for_manual_promotion,
        "metrics": metrics,
        "extra_criteria": EXTRA_STAGE_CRITERIA[stage],
        "extra_criteria_failures": extra_failures,
        "canonical_reasons": list(report.reasons),
    }
    write_json(stage_dir / "automatic_promotion_decision.json", decision)
    print("Held-out diagnostics:", json.dumps(metrics, indent=2))
    print("Automatic promotion eligible:", automatic_eligible)
    for reason in (*report.reasons, *extra_failures):
        print(" -", reason)
    return automatic_eligible, decision


## 6. Start TensorBoard (optional)

One root-level TensorBoard instance discovers all stage child directories.

In [ ]:
if START_TENSORBOARD:
    from tensorboard import notebook as tb_notebook

    tb_notebook.start(f'--logdir "{CURRICULUM_ROOT}"')
else:
    print("TensorBoard disabled; set START_TENSORBOARD=True to enable it.")


## 7. Train, gate, and advance

This loop has no retry path. A failed gate is an orderly stop, not an invitation to train more against the same canonical suite. The current stage's passing run directory becomes the sole warm-start source for the next stage.

In [ ]:
records = []
warm_start = None
curriculum_status = "running"

for stage in STAGES_TO_RUN:
    recipe_name = STAGE_RECIPES[stage]
    stage_dir = CURRICULUM_ROOT / f"stage_{stage}_{recipe_name}"
    stage_dir.mkdir(parents=True, exist_ok=False)
    source_run_dir = None if warm_start is None else str(warm_start.source_run_dir)
    print(f"\n===== Stage {stage}: {recipe_name} =====")
    cfg = make_stage_config(stage, stage_dir, warm_start)
    run_oracle_preflight(stage, cfg)

    final_model = train(cfg)
    del final_model
    render_and_audit_stage(cfg, stage_dir)
    automatic_eligible, decision = evaluate_for_promotion(stage, cfg, stage_dir)

    expected_evidence = [
        stage_dir / "promotion_report.json",
        stage_dir / "automatic_promotion_decision.json",
        *(stage_dir / f"held_out_stage_{index}.json" for index in range(stage + 1)),
    ]
    missing_evidence = [str(path) for path in expected_evidence if not path.is_file()]
    if missing_evidence:
        raise FileNotFoundError(f"Missing promotion evidence: {missing_evidence}")

    records.append(
        {
            "stage": stage,
            "recipe": recipe_name,
            "run_dir": str(stage_dir),
            "warm_start_source_run_dir": source_run_dir,
            "automatic_eligible": automatic_eligible,
            "decision_path": str(stage_dir / "automatic_promotion_decision.json"),
            "metrics": decision["metrics"],
        }
    )
    curriculum_status = "running" if automatic_eligible else "stopped_at_gate"
    write_json(
        CURRICULUM_ROOT / "curriculum_manifest.json",
        {
            "status": curriculum_status,
            "algorithm": RESOLVED_ALGO,
            "automatic_advancement": AUTO_ADVANCE,
            "stages_requested": list(STAGES_TO_RUN),
            "one_shot_canonical_evaluation": True,
            "records": records,
        },
    )

    if not automatic_eligible:
        print(f"STOP: Stage {stage} did not pass every promotion criterion.")
        break
    if stage != STAGES_TO_RUN[-1]:
        warm_start = WarmStartConfig(source_run_dir=str(stage_dir))
        print(f"PASS: Stage {stage} initializes Stage {stage + 1}.")
else:
    curriculum_status = "completed_requested_stages"

write_json(
    CURRICULUM_ROOT / "curriculum_manifest.json",
    {
        "status": curriculum_status,
        "algorithm": RESOLVED_ALGO,
        "automatic_advancement": AUTO_ADVANCE,
        "stages_requested": list(STAGES_TO_RUN),
        "one_shot_canonical_evaluation": True,
        "records": records,
    },
)
print("Curriculum status:", curriculum_status)
print("Manifest:", CURRICULUM_ROOT / "curriculum_manifest.json")


## 8. Interpret the result

completed_requested_stages means each requested stage passed using the checkpoint trained immediately before its one canonical evaluation. stopped_at_gate is a normal outcome: the run directory contains the model, plots, video, held-out episodes, reasons, diagnostics, and lineage needed for analysis.

Do not treat passing Stages 0–2 as evidence that free-standing two-humanoid tennis is solved. It establishes only that a transferred constrained policy met these fixed curriculum contracts.

## 9. Disconnect Colab

The default releases the Colab runtime after either all requested stages pass or an orderly gate failure is recorded. Unexpected exceptions leave the runtime connected so the traceback and partial artifacts can be debugged.

In [ ]:
if DISCONNECT_WHEN_DONE:
    from courtside_dynamics.notebook_utils import disconnect_runtime

    disconnect_runtime(delay_seconds=30)
else:
    print("Runtime left connected for inspection.")
